In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

# add project root (parent of current file) to path
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
             
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scripts.TPS import ThinPlateSpline
from scripts.plotting import *

# ---------- Load vector field ----------
def load_vector_field(csv_path):
    df = pd.read_csv(csv_path)
    X = df[["x", "y"]].values
    V = df[["vx", "vy"]].values
    time = df["time"].values
    return X, V, time

# ---------- File organization ----------
path_map = {
    "straight_line": "./data/1d/straight_line.csv",
    "sine_curve": "./data/1d/sine_curve.csv",
    "branch_2": "./data/1d/branch_2.csv",
    "branch_4": "./data/1d/branch_4.csv",
    "rotation": "./data/2d/rotation.csv",
    "spiral": "./data/2d/spiral.csv",
    "saddle": "./data/2d/saddle.csv",
    "quadratic_source_sink": "./data/2d/quadratic_source_sink.csv"
}

# Custom layout
row1_names = ["straight_line", "sine_curve", "branch_2", "branch_4"]
row2_names = ["rotation", "spiral", "saddle", "quadratic_source_sink"]
plot_order = row1_names + row2_names

fig, axs = plt.subplots(1, 8, figsize=(32, 4))

for ax, name in zip(axs, plot_order):
    X, V, time = load_vector_field(path_map[name])

    # normalize color to [0,1] for consistent colormap
    time = (time - time.min()) / (time.max() - time.min() + 1e-12)

    # Fit TPS *only* to ground-truth vector field
    tps_vf = ThinPlateSpline(X, n_control_points=100)
    tps_vf.fit(V, dof=15)

    # ---- panel-specific layout ----
    if name == "straight_line":
        stream_density, aspect = 0.2, 2.5
    elif name == "sine_curve":
        stream_density, aspect = 0.6, 2.0
    elif name == "branch_2":
        stream_density, aspect = 0.6, 1.5
    elif name == "branch_4":
        stream_density, aspect = 0.8, 1.5
    else:
        stream_density, aspect = 0.6, "equal"

    # ---- plot ----
    plot_velocity_streamplot(
        X_2d=X,
        tps_vf=tps_vf,
        grid_density=1.0,
        stream_density=stream_density,
        scatter_color=time,
        scatter_size=400,
        scatter_alpha=0.2,
        ax=ax,
        # title=name.replace("_", " "),
        aspect=aspect,
        vmin=0.0,
        vmax=1.0,
        arrowsize=3.0,
        cmap="viridis",
        show_axes=False,
        streamline_thickness=4.0,
        grid_size=50,
        pad_frac=0.05
    )

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.manifold import MDS
from scipy.spatial.distance import squareform
from scripts.phase_distance_solver import *
import matplotlib.pyplot as plt
import numpy as np
import anndata
import umap
import scvelo as scv
from sklearn.manifold import TSNE
from scripts.phase_distance_solver import *

np.random.seed(42)

# ------------------------------------------
# simulate noisy data (your block unchanged)
# ------------------------------------------
simulation_results = {}
noise, extra_dim = 0.3, 5
for name, path in path_map.items():
    X_gt, V_gt, time = load_vector_field(path)
    X_noisy = X_gt + np.random.normal(scale=noise, size=X_gt.shape)
    V_noisy = V_gt + np.random.normal(scale=noise, size=V_gt.shape)
    X_dummy = np.random.normal(scale=noise, size=(X_gt.shape[0], extra_dim))
    V_dummy = np.random.normal(scale=noise, size=(V_gt.shape[0], extra_dim))
    X = np.hstack([X_noisy, X_dummy])
    V = np.hstack([V_noisy, V_dummy])
    simulation_results[name] = dict(X=X, V=V, X_gt=X_gt, V_gt=V_gt, true_time=time)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata
import scvelo as scv

from rpy2 import robjects as ro
from rpy2.robjects import conversion
from rpy2.robjects import numpy2ri, pandas2ri
from rpy2.robjects.vectors import StrVector

from scripts.evaluation import evaluate_embedding_method


# ------------------------------------------
# PLOT + SCORE
# ------------------------------------------
fig, axes = plt.subplots(1, 8, figsize=(32, 4), constrained_layout=True)
axes = axes.flatten()

records = []

for ax, (name, result) in zip(axes, simulation_results.items()):
    X, V        = result["X"], result["V"]
    X_gt, V_gt  = result["X_gt"], result["V_gt"]
    time        = result["true_time"]

    proj = X + V
    cell_names = [f"cell_{i}" for i in range(X.shape[0])]

    # ------------------------------------------
    # R / VELOVIZ (ALL INSIDE ONE CONTEXT)
    # ------------------------------------------
    with conversion.localconverter(
        ro.default_converter + numpy2ri.converter + pandas2ri.converter
    ):
        ro.globalenv["curr"] = X.T
        ro.globalenv["proj"] = proj.T
        ro.globalenv["cell_names"] = StrVector(cell_names)

        ro.r("""
        suppressPackageStartupMessages(library(veloviz))

        colnames(curr) <- cell_names
        colnames(proj) <- cell_names

        vv <- buildVeloviz(
            curr, proj,
            normalize.depth = FALSE,
            use.ods.genes = FALSE,
            alpha = 1,
            pca = FALSE,
            center = FALSE,
            scale = FALSE,
            k = 5,
            similarity.threshold = 0.25,
            distance.weight = 1,
            distance.threshold = 0.5,
            weighted = FALSE,
            verbose = FALSE
        )

        veloviz_embedding <- vv$fdg_coords
        cell_names_used   <- rownames(vv$fdg_coords)
        """)

        emb = np.array(ro.r["veloviz_embedding"])
        used_names = list(ro.r["cell_names_used"])

    # ------------------------------------------
    # CLEANUP AFTER R
    # ------------------------------------------
    keep_idx = [int(s.split("_")[-1]) for s in used_names]

    X_conn, V_conn = X[keep_idx], V[keep_idx]
    X_gt_conn, V_gt_conn = X_gt[keep_idx], V_gt[keep_idx]

    time_conn = (time[keep_idx] - time.min()) / (time.max() - time.min())

    # ------------------------------------------
    # scVelo projection (don’t overthink)
    # ------------------------------------------
    adata = anndata.AnnData(X_conn)
    adata.layers["position"] = X_conn
    adata.layers["velocity"] = V_conn
    adata.obsm["X_veloviz"]  = emb
    adata.obs["time"]        = time_conn

    scv.pp.neighbors(adata, use_rep="X", n_neighbors=30)
    scv.tl.velocity_graph(adata, xkey="position", vkey="velocity")
    scv.tl.velocity_embedding(adata, basis="veloviz")

    V_emb = adata.obsm["velocity_veloviz"]

    # ------------------------------------------
    # PLOT (NO ANNOTATIONS)
    # ------------------------------------------
    scv.pl.velocity_embedding_stream(
        adata,
        basis="veloviz",
        color="time",
        cmap="viridis",
        ax=ax,
        show=False,
        legend_loc=None,
        colorbar=False,
        density=0.3,
        arrow_size=2.5,
        linewidth=3.0,
        alpha=0.15,
        size=1000,
    )

    ax.set_title(name)
    ax.set_aspect("equal")
    ax.axis("off")

    # ------------------------------------------
    # METRICS (WHATEVER IT SPITS OUT)
    # ------------------------------------------
    scores = evaluate_embedding_method(
        X_gt_conn, emb,
        V_gt_conn, V_emb,
        k=30
    )
    scores["dataset"] = name
    records.append(scores)

fig_dir = "./figures/simulation"
os.makedirs(fig_dir, exist_ok=True)

fig_path = os.path.join(fig_dir, "veloviz_embedding_streams.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Saved figure to {fig_path}")
plt.show()

# ------------------------------------------
# SAVE SCORES
# ------------------------------------------
df_scores = pd.DataFrame(records).set_index("dataset")
os.makedirs("./data/8_vf_collection", exist_ok=True)
df_scores.to_csv("./data/8_vf_collection/veloviz.csv")

print("\nVeloviz scores saved to ./data/8_vf_collection/veloviz.csv\n")
print(df_scores.round(4))